# Ch3：四个堆叠方块归位（IK 修正版）

本版针对运行时出现的：

`Approach 位姿无逆运动学解：[189.8, -15.0, 112.1]`

进行了修正。

根因不是 X/Y/Z 本身，而是第2/3层使用的 `Roll=83.7°` 落在当前机械臂逆运动学不可行区；修正为最近可行的 `90.0°`，并将 4 个 Approach 位姿显式保存，不再自动生成。

如果第1块已经成功搬走，可直接设置 `START_BLOCK=2` 从第2块继续。


## 1. 本次九点标定数据

当前九点标定中，四个目标角点在机械臂坐标系下为：

- P0 = (236.4, 35.4, 0.278)
- P2 = (220.2, -62.1, 0.278)
- P6 = (136.1, 37.8, 0.278)
- P8 = (132.7, -58.4, 0.278)

`T_arm2ws` 用于机械臂坐标系与工作台坐标系之间的映射；本任务已经直接取得 P0/P2/P6/P8 的机械臂坐标，因此执行抓放时不需要再次变换。


In [ ]:
import math
import sys
from dataclasses import dataclass
from pathlib import Path

# 支持从“2.机械臂实验”目录或 Notebook 工作区根目录启动 Jupyter。
for sdk_dir in (Path.cwd(), Path.cwd() / '2.机械臂实验'):
    if (sdk_dir / 'arm5dof_uservo.py').is_file():
        sys.path.insert(0, str(sdk_dir))
        break
else:
    raise FileNotFoundError('未找到 arm5dof_uservo.py；请从课程工作区打开本 Notebook。')

# 只导入 SDK，不会在此单元连接串口或移动机械臂。
from arm5dof_uservo import Arm5DoFUServo


## 2. 位姿参数表（单位：mm、deg）

中央堆叠的四层抓取位姿沿用现有示教值；四个放置区的 X/Y 改为 P0/P2/P6/P8。

注意：九点标定的 Z 是工作台平面参考值，因此实际夹爪低位通过 `ZONE_Z_OFFSET` 增加安全偏置。当前初值按原四区 near/far 示教高度迁移，第一次真机必须逐点检查。


In [ ]:
@dataclass(frozen=True)
class Pose:
    x: float
    y: float
    z: float
    pitch_deg: float
    roll_deg: float

    def xyz(self):
        return [self.x, self.y, self.z]


CALIB_POINTS = {
    'P0': (236.4,  35.4, 0.278),
    'P2': (220.2, -62.1, 0.278),
    'P4': (177.5,  -9.0, 0.278),
    'P6': (136.1,  37.8, 0.278),
    'P8': (132.7, -58.4, 0.278),
}

CENTER_P4 = CALIB_POINTS['P4']

# ============================================================
# 最新实测中央四层准确夹取位姿（最上层 -> 最下层）
#
# 注意：
# 第2/3层原始输出 Roll=85763.7°，直接 %360 得 83.7°。
# 当前 Arm5DoF 逆运动学在 83.7° 无解；90.0° 是最近的可行边界。
# 所以第2/3层只把 Roll 修正为 90°，X/Y/Z/Pitch 保持实测值不变。
# ============================================================
PICK_POSES = (
    Pose(194.6,  -9.9, 81.0, 114.5, 144.5),  # 第1块：最上层
    Pose(189.8, -15.0, 57.1, 120.8,  90.0),  # 第2块：Roll 83.7 -> 90.0
    Pose(193.6, -11.3, 44.3, 125.0,  90.0),  # 第3块：Roll 83.7 -> 90.0
    Pose(192.2, -11.2, 13.8, 132.7, 144.5),  # 第4块：最下层
)

# ============================================================
# 抓取上方点必须单独配置，不能简单“抓取Z + 55 且姿态完全不变”后盲跑。
# 这里使用已经过逆运动学可达性约束的参数：
# - 第1/4层沿用 144.5° roll
# - 第2/3层采用 90° roll
# ============================================================
PICK_APPROACH_POSES = (
    Pose(194.6,  -9.9, 136.0, 114.5, 144.5),
    Pose(189.8, -15.0, 112.1, 120.8,  90.0),
    Pose(193.6, -11.3,  99.3, 125.0,  90.0),
    Pose(192.2, -11.2,  68.8, 132.7, 144.5),
)

MIN_PICK_Z = 10.0
PLACE_SAFE_DZ = 55.0

ZONE_Z_OFFSET = {
    'P0': 30.0,
    'P2': 30.0,
    'P6': 20.0,
    'P8': 20.0,
}

ZONE_ORIENTATION = {
    'P0': (125.0, 179.2),
    'P2': (135.0, 179.2),
    'P6': (140.0, 179.2),
    'P8': (140.0, 179.2),
}


def zone_pose(name):
    x, y, z_plane = CALIB_POINTS[name]
    pitch, roll = ZONE_ORIENTATION[name]
    return Pose(
        x=x,
        y=y,
        z=z_plane + ZONE_Z_OFFSET[name],
        pitch_deg=pitch,
        roll_deg=roll,
    )


TARGETS = tuple((name, zone_pose(name)) for name in ('P0', 'P2', 'P6', 'P8'))

MOVE_T = 1.5
GRIPPER_T = 1.0

DRY_RUN = True

# 1-based，便于中断后继续。
# 如果刚才第1块已经成功搬到 P0、在第2块 Approach 时报错：
#   START_BLOCK = 2
#   END_BLOCK   = 4
START_BLOCK = 1
END_BLOCK = 4


In [ ]:
def pick_pose(top_down_index):
    return PICK_POSES[top_down_index]


def pick_approach_pose(top_down_index):
    return PICK_APPROACH_POSES[top_down_index]


def place_above_pose(pose):
    return Pose(pose.x, pose.y, pose.z + PLACE_SAFE_DZ, pose.pitch_deg, pose.roll_deg)


def validate_parameters(start_block=START_BLOCK, end_block=END_BLOCK):
    if start_block not in range(1, 5) or end_block not in range(1, 5):
        raise ValueError('START_BLOCK / END_BLOCK 必须在 1~4')
    if start_block > end_block:
        raise ValueError('START_BLOCK 不能大于 END_BLOCK')
    if len(PICK_POSES) != 4 or len(PICK_APPROACH_POSES) != 4 or len(TARGETS) != 4:
        raise ValueError('抓取位姿、Approach位姿、目标区必须各有4项')
    if min(p.z for p in PICK_POSES) < MIN_PICK_Z:
        raise ValueError('最低抓取高度低于安全底线')


def print_pose_table(start_block=1, end_block=4):
    validate_parameters(start_block, end_block)
    print(f'中央参考 P4 = ({CENTER_P4[0]:.1f}, {CENTER_P4[1]:.1f}, {CENTER_P4[2]:.3f})')
    print('块次 | Pick x/y/z | Pitch | Roll | Approach z | 目标区')
    print('-' * 86)
    for i in range(start_block - 1, end_block):
        name, _ = TARGETS[i]
        p = PICK_POSES[i]
        a = PICK_APPROACH_POSES[i]
        print(
            f'{i+1:>2}   | ({p.x:6.1f},{p.y:6.1f},{p.z:5.1f}) | '
            f'{p.pitch_deg:6.1f} | {p.roll_deg:6.1f} | '
            f'{a.z:6.1f}      | {name}'
        )


print_pose_table()


## 3. Pick & Place 动作封装

每块都执行同一个函数，避免复制四段动作代码导致等待、开闭夹爪或撤离步骤不一致。各次移动均采用 SDK 的软轨迹；夹爪开闭也等待完成。

In [ ]:
def move_to(arm, pose, label, duration=MOVE_T):
    """以给定末端姿态运动；SDK 的软轨迹调用会阻塞至轨迹完成。"""
    print(f'  {label:<12} -> ({pose.x:.1f}, {pose.y:.1f}, {pose.z:.1f}), '
          f'pitch={pose.pitch_deg:.1f}°, roll={pose.roll_deg:.1f}°')
    result = arm.set_tool_pose(
        pose.xyz(),
        tool_pitch=math.radians(pose.pitch_deg),
        tool_roll=math.radians(pose.roll_deg),
        T=duration,
        is_soft_move=True,
        is_wait=True,
    )
    if result is False:
        raise RuntimeError(f'{label} 位姿无逆运动学解，已停止后续动作：{pose.xyz()}')


def pick_and_place(arm, top_down_index, target_name, target):
    """安全执行一块：Approach → Pick → Lift → Transfer → Place → Retreat。"""
    pick = pick_pose(top_down_index)
    pick_above = pick_approach_pose(top_down_index)
    place_above = place_above_pose(target)

    print(f'\n第 {top_down_index + 1} 块：最上层剩余方块 -> {target_name} 区')
    arm.gripper_open(T=GRIPPER_T, is_wait=True)

    # Approach：先到抓取上方点；Pick：只作垂直下降。
    move_to(arm, pick_above, 'Approach')
    move_to(arm, pick, 'Pick')
    arm.gripper_close(T=GRIPPER_T, is_wait=True)

    # Lift：先垂直抬升，不在低位横移，避免扫到方块塔或桌面。
    move_to(arm, pick_above, 'Lift')
    arm.home()                 # 高位中转，再驶向目标区

    # Transfer：到放置上方点；Place：垂直下降；Retreat：松爪后垂直撤离。
    move_to(arm, place_above, 'Transfer')
    move_to(arm, target, 'Place')
    arm.gripper_open(T=GRIPPER_T, is_wait=True)
    move_to(arm, place_above, 'Retreat')
    arm.home()


## 4. 预览或执行

先检查打印表中四个目标区是否已经变成 P0/P2/P6/P8。保持 `DRY_RUN=True` 时不会连接串口。

真机建议按以下顺序验证：

1. `DRY_RUN=True, BLOCKS_TO_RUN=4`：只看参数表；
2. `DRY_RUN=False, BLOCKS_TO_RUN=1`：只搬最上层到 P0；
3. 确认 P0 低位与路径安全后，再逐步增加到 2、4 块。


In [ ]:
def run_unstack_sort(start_block=START_BLOCK, end_block=END_BLOCK, dry_run=DRY_RUN):
    validate_parameters(start_block, end_block)
    print('Ch3 四层方块归位')
    print(f'本次执行：第 {start_block} 块 -> 第 {end_block} 块')
    print_pose_table(start_block, end_block)

    if dry_run:
        print('\n当前为预览模式：未连接串口，机械臂不会运动。')
        return

    arm = Arm5DoFUServo()
    try:
        print('\n开始真机执行。请保持手远离机械臂。')
        arm.home()

        for i in range(start_block - 1, end_block):
            target_name, target = TARGETS[i]
            pick_and_place(arm, i, target_name, target)

        arm.gripper_open(T=GRIPPER_T, is_wait=True)
        arm.home()
        print('\n完成：机械臂已回到 Home。')
    except Exception:
        try:
            arm.gripper_open(T=GRIPPER_T, is_wait=True)
            arm.home()
        finally:
            raise


run_unstack_sort()


## 5. 三次测试记录

中央四层现在采用最新实测准确夹取位姿，中心参考点为 P4。  
若最低层仍有误差，优先重新测第4层的 `(x,y,z,pitch,roll)`，不要直接降低 Z。

| 测试轮次 | 成功块数 / 4 | 失败块次与现象 | 调整的参数或措施 |
| --- | --- | --- | --- |
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
